# ML-07 — Baseline Action Score and Top-10 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane 2 continuation** (refresh/opportunity-scoring). Before coding anything, I check two
signals my rule idea leans on — the same way the reference pipeline's flags
(`stale_visible_page`, `low_ctr_visible_page` in `scripts/02_baseline_score.py`) are each backed
by a real, checkable pattern rather than a guess.

- **Signal A — staleness**, behind FlyRank's refresh flag (`days_since_last_update >= 180`
  drives `stale_visible_page`). Bucketed via the pre-built `freshness_tier` column.
- **Signal B — CTR-vs-position**, behind the CTR-fix flag (`low_ctr_visible_page`: a CTR that's
  low *for that page's ranking position*, not low in absolute terms). Bucketed by whether a
  page's CTR sits below the median CTR of other pages at the same `position_tier`.

Both are checked against `is_declining_label` (the Lane-2 proxy target) — **as an outcome to
verify against, never as a rule input.** `trend_direction` / `trend_pct` / `is_declining_label`
stay out of the score itself.

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# is_declining_label isn't in the raw CSV -- built the same way the pipeline does.
# Used below ONLY to verify signals and evaluate the rule -- never as a score input.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"{len(df):,} rows | base decline rate: {base_rate:.3f}")

# --- Signal A: staleness, behind the refresh flag (days_since_last_update >= 180) ---
tier_order_fresh = ["never", "0-30", "31-90", "91-180", "181+"]
signal_a = df.groupby("freshness_tier")["is_declining_label"].agg(n="count", decline_rate="mean")
signal_a = signal_a.reindex([t for t in tier_order_fresh if t in signal_a.index])
print("\nSignal A -- staleness (freshness_tier) vs decline rate:")
print(signal_a)

30,000 rows | base decline rate: 0.542

Signal A -- staleness (freshness_tier) vs decline rate:
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
31-90             175      0.588571
91-180           9171      0.611057
181+              174      0.471264


**Verdict A: MIXED.** The 91–180 bucket (n=9,171 — the second-largest, reliable) does run hot
(0.611 vs 0.542 base), consistent with "staler pages decline more." But it isn't monotonic: the
*most*-stale bucket, 181+ (n=174), comes in at 0.471 — **below** base rate, the opposite of what
the refresh flag's own threshold (`>=180`) assumes. That tail bucket is small and noisy, so I
won't call it OPPOSITE, but it clearly doesn't CONFIRM either. Staleness alone is not a reliable
driver — I will not build the score on it alone.

In [2]:
# --- Signal B: CTR-vs-position, behind the CTR-fix flag (low CTR *for that position*) ---

# Gotcha from the data dictionary: avg_position == 0 means "no position data", not rank
# zero -- and this slice's own position_tier column bins all 1,205 of those rows into
# "top_3" (0 <= 3). Left in, they'd fake an ultra-low "top_3" decline rate. Excluded here.
has_position = df["avg_position"] > 0
print(f"rows with real position data: {has_position.sum():,} of {len(df):,} "
      f"({(~has_position).sum():,} excluded as no-position-data)")

sub = df.loc[has_position].copy()
tier_median_ctr = sub.groupby("position_tier")["ctr"].transform("median")
sub["ctr_underperforms_position"] = (sub["ctr"] < tier_median_ctr).astype(int)

signal_b = sub.groupby("ctr_underperforms_position")["is_declining_label"].agg(n="count", decline_rate="mean")
print("\nSignal B -- CTR below its position tier's median vs decline rate:")
print(signal_b)

rows with real position data: 28,795 of 30,000 (1,205 excluded as no-position-data)

Signal B -- CTR below its position tier's median vs decline rate:
                                n  decline_rate
ctr_underperforms_position                     
0                           15716      0.540659
1                           13079      0.593088


**Verdict B: CONFIRMED.** Pages whose CTR sits below the median for their own position tier
decline noticeably more often (0.593 vs 0.541, both large buckets — 13,079 and 15,716 rows, not
a handful of outliers). This matches the CTR-fix flag's core assumption: what matters is CTR
*relative to peers at the same ranking position*, not CTR in isolation — and that relative gap
does associate with higher decline rates here.

**The rule, in plain words:** *"A page is worth a CTR-fix review if it gets enough search
visibility to matter, and its click-through rate is worse than other pages ranking in the same
position band — a fixable, actionable gap, not just an inherently low-ceiling query."*

Staleness (MIXED) is left out of the score entirely — a clearly negative signal check just
saved the rule from leaning on something that doesn't hold up past its own threshold zone.
The rule instead gates on two things I can defend: **visibility** (`impressions_90d >= 500`,
matching the reference pipeline's own "worth reviewing" bar) and **Signal B** (CONFIRMED,
above).

**Score, v1 → v2 (a revision caught by checking precision@10, not by guessing):** I first
ranked flagged pages by raw `impressions_90d` ("more traffic = bigger opportunity"). That
scored precision@10 = 0.400 — *below* the 0.542 base rate — because sorting by traffic size
pulled giant, structurally low-risk pages (the `excellent` impression tier actually declines
*less* than the `moderate` tier, a real reversal) ahead of smaller pages with a far worse CTR
problem. The fix: rank flagged pages by **`ctr_gap = tier_median_ctr − ctr`** — how *severe*
the underperformance is, not how big the audience is. Subtraction (not a ratio) so it stays
defined even in position tiers whose median CTR is exactly 0.

**Reason codes** (exactly one per row, mutually exclusive):
- `low_ctr_for_position` — has position data, visible, and CTR underperforms its tier → flagged.
- `ctr_healthy_for_position` — has position data and visible, but CTR is at/above tier median.
- `not_visible_enough` — has position data but `impressions_90d < 500` → not worth a slot.
- `no_position_data` — `avg_position == 0`; can't judge a CTR gap without a position to compare
  against.

**Action labels:** `low_ctr_for_position` → `review_ctr_fix`; everything else → `monitor` or
`no_action` (below), never review — the queue should send reviewers only where the rule has
actual evidence, not a default action for every row.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
from pathlib import Path

VISIBLE_MIN_IMPRESSIONS = 500  # matches the reference pipeline's own "worth reviewing" bar

has_position = df["avg_position"] > 0
visible = df["impressions_90d"] >= VISIBLE_MIN_IMPRESSIONS

# Recompute the tier-median CTR gap over the FULL frame (section 1 used a filtered copy).
tier_median_ctr_full = pd.Series(np.nan, index=df.index)
tier_median_ctr_full.loc[has_position] = df.loc[has_position].groupby("position_tier")["ctr"].transform("median")
ctr_underperforms = pd.Series(False, index=df.index)
ctr_underperforms.loc[has_position] = df.loc[has_position, "ctr"] < tier_median_ctr_full.loc[has_position]

flagged = has_position & visible & ctr_underperforms


def reason_code(i: int) -> str:
    if not has_position.iat[i]:
        return "no_position_data"
    if not visible.iat[i]:
        return "not_visible_enough"
    if ctr_underperforms.iat[i]:
        return "low_ctr_for_position"
    return "ctr_healthy_for_position"


ACTION_BY_REASON = {
    "low_ctr_for_position": "review_ctr_fix",
    "not_visible_enough": "no_action",
    "ctr_healthy_for_position": "monitor",
    "no_position_data": "monitor",
}

df["reason_code"] = [reason_code(i) for i in range(len(df))]
df["action"] = df["reason_code"].map(ACTION_BY_REASON)

# Score, v2: rank flagged pages by CTR-GAP SEVERITY (how far below its position tier's
# median a page's CTR sits), not by raw impressions_90d. v1 (impressions-based) scored
# precision@10 = 0.400, BELOW the 0.542 base rate -- sorting by traffic size pulled in
# giant, low-risk "excellent"-tier pages ahead of smaller pages with a much worse CTR
# gap. Subtraction (not a ratio) keeps this defined even where a tier's median CTR is 0.
df["ctr_gap"] = tier_median_ctr_full - df["ctr"]
df["score"] = np.where(flagged, df["ctr_gap"], 0)

print("reason_code counts:")
print(df["reason_code"].value_counts())
print("\naction counts:")
print(df["action"].value_counts())

queue = df.sort_values(["score", "impressions_90d"], ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

output_cols = [
    "rank", "content_id", "client_id", "action", "reason_code", "score", "ctr_gap",
    "impressions_90d", "avg_position", "ctr", "days_since_last_update",
    "is_declining_label",
]
out_path = Path("../outputs/baseline_action_score.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_cols].to_csv(out_path, index=False)

print(f"\nwrote {len(queue):,} rows to {out_path}")
print(f"flagged (low_ctr_for_position): {flagged.sum():,} rows")

reason_code counts:
reason_code
not_visible_enough          12069
ctr_healthy_for_position    11753
low_ctr_for_position         4973
no_position_data             1205
Name: count, dtype: int64

action counts:
action
monitor           12958
no_action         12069
review_ctr_fix     4973
Name: count, dtype: int64



wrote 30,000 rows to ..\outputs\baseline_action_score.csv
flagged (low_ctr_for_position): 4,973 rows


**Evaluation note:** `is_declining_label` appears in the output CSV and in the check below only
to score the queue after the fact — it was never read by the `score`/`reason_code`/`action`
logic above. Precision@K is the honest metric for a fixed-capacity queue (per
`building-baselines`): of the top K rows the rule sends a reviewer, how many were actually
right?

In [4]:
def precision_at_k(labels: pd.Series, k: int) -> float:
    return labels.head(k).mean()

for k in (10, 50):
    p = precision_at_k(queue["is_declining_label"], k)
    print(f"precision@{k}: {p:.3f}  (base rate: {base_rate:.3f})")

precision@10: 0.800  (base rate: 0.542)
precision@50: 0.860  (base rate: 0.542)


## 3. Top-10 review

*For each of my top ten: the action, why it's there, and what would make it wrong.*

In [5]:
def wrongness_note(main_intent: str) -> str:
    notes = {
        "informational": (
            "wrong if searchers get their answer straight from the SERP snippet -- "
            "informational intent structurally suppresses CTR even on a healthy page"
        ),
        "navigational": (
            "wrong if this is a branded/navigational query -- users already know the "
            "destination and skip the snippet, so low CTR may be structural, not a content flaw"
        ),
        "transactional": (
            "wrong if a SERP feature (ads, shopping carousel) above this result is eating "
            "clicks -- the fix would be off-page, not on-page"
        ),
        "commercial": (
            "wrong if a SERP feature (ads, shopping carousel) above this result is eating "
            "clicks -- the fix would be off-page, not on-page"
        ),
    }
    return notes.get(
        main_intent,
        "wrong if intent is unrecorded -- no real baseline to judge this CTR against",
    )


top10 = queue.head(10).copy()
top10["tier_median_ctr"] = top10.apply(
    lambda r: tier_median_ctr_full.loc[df["content_id"] == r["content_id"]].iloc[0], axis=1
)

print(f"Top 10 of {len(queue):,} -- {flagged.sum():,} rows carry reason_code=low_ctr_for_position\n")
for _, row in top10.iterrows():
    print(
        f"{int(row['rank']):>2}. {row['content_id']} ({row['client_id']}) "
        f"-> action={row['action']}  reason={row['reason_code']}"
    )
    print(
        f"    why: {row['impressions_90d']:,.0f} impressions/90d at avg position "
        f"{row['avg_position']:.1f}, CTR {row['ctr']:.2f}% vs a {row['tier_median_ctr']:.2f}% "
        f"median CTR for its position tier -- visible traffic, underperforming click rate"
    )
    print(f"    what would make it wrong: {wrongness_note(row['main_intent'])}")
    print()

Top 10 of 30,000 -- 4,973 rows carry reason_code=low_ctr_for_position

 1. content_c8e9d6ab9013 (client_19581e27de) -> action=review_ctr_fix  reason=low_ctr_for_position
    why: 208,678 impressions/90d at avg position 9.7, CTR 0.00% vs a 0.16% median CTR for its position tier -- visible traffic, underperforming click rate
    what would make it wrong: wrong if searchers get their answer straight from the SERP snippet -- informational intent structurally suppresses CTR even on a healthy page

 2. content_f986bd514b6e (client_7f2253d7e2) -> action=review_ctr_fix  reason=low_ctr_for_position
    why: 22,456 impressions/90d at avg position 6.6, CTR 0.00% vs a 0.16% median CTR for its position tier -- visible traffic, underperforming click rate
    what would make it wrong: wrong if intent is unrecorded -- no real baseline to judge this CTR against

 3. content_825a9788af8d (client_4e07408562) -> action=review_ctr_fix  reason=low_ctr_for_position
    why: 16,786 impressions/90d at avg posi

**Sanity check on the v2 sort key.** `ctr_gap` is capped: since `ctr` can never go below 0, the
biggest possible gap in any position tier is that tier's own median — so every page with
`ctr == 0` in the tier with the highest median CTR ties at the single largest `ctr_gap` value in
the whole dataset. That's exactly the group that filled the top 10 above, and within a tied
block, `impressions_90d` (the secondary sort key) is doing the actual ordering again — the same
mechanism that hurt v1. Before trusting precision@10 = 0.800, the real question is whether that
result depends on *which* rows within the tie happened to have the most traffic, or whether the
whole tied group is uniformly high-risk regardless of tie-break.

In [6]:
max_gap = queue.loc[flagged, "ctr_gap"].max()
tied_at_max = flagged & (df["ctr_gap"] == max_gap)

print(f"max ctr_gap among flagged rows: {max_gap:.3f}")
print(f"rows tied at that maximum: {tied_at_max.sum():,} (of {flagged.sum():,} flagged)")
print(f"decline rate WITHIN the entire tied group: {df.loc[tied_at_max, 'is_declining_label'].mean():.3f}")
print(f"decline rate across ALL flagged rows (for comparison): {df.loc[flagged, 'is_declining_label'].mean():.3f}")
print(f"precision@10 from the actual queue: {precision_at_k(queue['is_declining_label'], 10):.3f}")

max ctr_gap among flagged rows: 0.160


rows tied at that maximum: 491 (of 4,973 flagged)
decline rate WITHIN the entire tied group: 0.823
decline rate across ALL flagged rows (for comparison): 0.662
precision@10 from the actual queue: 0.800


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# --- Weak picks in the top 10 ---
weak = top10[top10["is_declining_label"] == 0]
print(f"{len(weak)} of the top 10 are NOT actually declining ({len(weak)/10:.0%}) -- "
      f"vs base rate {base_rate:.1%}.\n")
for _, row in weak.iterrows():
    print(
        f"rank {int(row['rank'])}: {row['content_id']} -- ctr_gap={row['ctr_gap']:.3f} "
        f"but is_declining_label=0. Even the worst CTR gaps aren't a perfect decline "
        f"predictor -- the CTR-vs-position signal was CONFIRMED, not deterministic; a ~5-9 "
        f"point lift in decline rate still leaves plenty of exceptions in any top slice."
    )

# --- Leakage check: the score must not depend on the label or its inputs ---
label_derived_cols = {"trend_direction", "trend_pct", "is_declining_label"}
check_df = df.drop(columns=[c for c in label_derived_cols if c in df.columns])

has_position_chk = check_df["avg_position"] > 0
visible_chk = check_df["impressions_90d"] >= VISIBLE_MIN_IMPRESSIONS
tier_median_chk = pd.Series(np.nan, index=check_df.index)
tier_median_chk.loc[has_position_chk] = (
    check_df.loc[has_position_chk].groupby("position_tier")["ctr"].transform("median")
)
ctr_underperforms_chk = pd.Series(False, index=check_df.index)
ctr_underperforms_chk.loc[has_position_chk] = (
    check_df.loc[has_position_chk, "ctr"] < tier_median_chk.loc[has_position_chk]
)
flagged_chk = has_position_chk & visible_chk & ctr_underperforms_chk
ctr_gap_chk = tier_median_chk - check_df["ctr"]
score_chk = np.where(flagged_chk, ctr_gap_chk, 0)

assert np.array_equal(score_chk, df["score"].to_numpy()), "score changed after dropping label-derived columns!"
print("\nLeakage check passed: rebuilding the score with trend_direction / trend_pct / "
      "is_declining_label dropped from the frame entirely produces an IDENTICAL score array. "
      "No future-window or label-derived column is a rule input.")

2 of the top 10 are NOT actually declining (20%) -- vs base rate 54.2%.

rank 5: content_5d5653c4eb4f -- ctr_gap=0.160 but is_declining_label=0. Even the worst CTR gaps aren't a perfect decline predictor -- the CTR-vs-position signal was CONFIRMED, not deterministic; a ~5-9 point lift in decline rate still leaves plenty of exceptions in any top slice.
rank 6: content_847a841969a2 -- ctr_gap=0.160 but is_declining_label=0. Even the worst CTR gaps aren't a perfect decline predictor -- the CTR-vs-position signal was CONFIRMED, not deterministic; a ~5-9 point lift in decline rate still leaves plenty of exceptions in any top slice.



Leakage check passed: rebuilding the score with trend_direction / trend_pct / is_declining_label dropped from the frame entirely produces an IDENTICAL score array. No future-window or label-derived column is a rule input.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.